In [87]:
#Load the data  and split 70/15/15

import torch
import torch.nn as nn  # Neural network layers
from torch.utils.data import Dataset, DataLoader  # Data handling
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# ── 1. Load and split data BEFORE normalizing ────────────────────────────────
df = pd.read_csv("digital_habits_vs_mental_health.csv").dropna()  # Load data, remove missing
X = df.drop("stress_level", axis=1).values.astype(np.float32)  # Features (all except target)
y = df["stress_level"].values.astype(np.float32)  # Target (continuous stress values)

# Split: 70% train, 15% val, 15% test (reproducible with random_state=0)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.30, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.5, random_state=0)

# ── 2. Normalize using TRAIN stats ONLY (prevents cheating) ───────────────────
mean = X_train.mean(axis=0)  # Average of each feature in training data
std = X_train.std(axis=0) + 1e-8  # Standard deviation + tiny number (avoid divide by 0)
X_train = (X_train - mean) / std  # Normalize train: mean=0, std=1
X_val = (X_val - mean) / std     # Apply SAME stats to validation
X_test = (X_test - mean) / std   # Apply SAME stats to test

# ── 3. PyTorch Dataset class (wraps your data) ────────────────────────────────
class TabularDataset(Dataset):  # Custom class for tabular data
    def __init__(self, X, y):  # Constructor
        self.X = torch.tensor(X, dtype=torch.float32)  # Convert features to tensor
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Targets as column [N,1]

    def __len__(self):  # How many samples?
        return len(self.y)

    def __getitem__(self, idx):  # Get one sample by index
        return self.X[idx], self.y[idx]  # Return features + target

# Create data loaders (automatic batching)
train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=64, shuffle=True)
# Training: 64 samples/batch, shuffle=better learning

val_loader = DataLoader(TabularDataset(X_val, y_val), batch_size=256)
# Validation: larger batches (no shuffle needed)


In [95]:
# Normalize the data and prepare for training
import torch
import torch.nn as nn  # Neural network building blocks
import torch.nn.functional as F  # Activation functions (ReLU)

class TabularNet(nn.Module):  # Our neural network class
                        # DROPOUT: 0.1 weak regularization|0.3 standard|0.5 strong
    def __init__(self, d_in, hidden, d_out, dropout=0.3):
        # __init__ = "constructor" - builds the layers when creating model
        super().__init__()  # Initialize PyTorch's base model class

        # Layer 1: Input features → First hidden layer
        self.fc1 = nn.Linear(d_in, hidden)      # Linear: multiplies weights + bias
        self.bn1 = nn.BatchNorm1d(hidden)       # BatchNorm: stabilizes training
        self.drop = nn.Dropout(dropout)         # Dropout: randomly ignores 30% neurons (prevents overfitting)

        # Layer 2: Hidden → Hidden (same size)
        self.fc2 = nn.Linear(hidden, hidden)    # Another hidden layer
        self.bn2 = nn.BatchNorm1d(hidden)       # BatchNorm again

        # Output layer: Hidden → Final prediction
        self.out = nn.Linear(hidden, d_out)     # d_out=1 for regression (stress level)

    def forward(self, x):  # "forward" = how data flows through the network
        # Input → Layer1 → Activation → Dropout
        x = F.relu(self.bn1(self.fc1(x)))  # Linear → BatchNorm → ReLU (0 if negative)
        x = self.drop(x)                   # Randomly drop neurons

        # Hidden → Layer2 → Activation → Dropout
        x = F.relu(self.bn2(self.fc2(x)))  # Same pattern
        x = self.drop(x)

        # Final output (raw stress level prediction)
        return self.out(x)  # No activation = continuous output for regression

# CREATE THE MODEL
d_in = X_train.shape[1]  # # of input features (from your data)
model = TabularNet(d_in=d_in, hidden=64, d_out=1)  # 10→64→64→1 neurons

# TRAINING SETUP                                  # started at 1e-3  |1e-5 light|1e-4 standard|1e-3 strong
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
# Adam: smart learning algorithm, lr=learning rate, weight_decay=prevents overfitting

#I will use hubber since I know the data has outliers
#criterion = nn.HuberLoss(delta = 1.0)  # Loss: Mean Squared Error (good for regression)
criterion = nn.MSELoss()  # Loss: Mean Squared Error (good for regression)
# Measures: (predicted_stress - true_stress)^2 average


In [96]:
# ── TRAINING LOOP ─────────────────────────────────────────────────────────────
n_epochs = 100  # Number of full passes through training data
train_losses, val_losses = [], []  # Track loss history

# Before training loop
class EarlyStopping:  # 5-aggressive stopping|10 standard| 20 slow
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience

early_stop = EarlyStopping(patience=10)

for epoch in range(n_epochs):
    # === TRAINING PHASE ===
    model.train()  # Enable dropout, batchnorm updates
    train_loss = 0
    for Xb, yb in train_loader:  # Batches of 64
        optimizer.zero_grad()     # Clear old gradients
        preds = model(Xb)         # Forward pass → predictions
        loss = criterion(preds, yb)  # MSE loss
        loss.backward()           # Compute gradients
        #this will prevent explosion from outliers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()          # Update weights
        train_loss += loss.item()  # Accumulate loss

    # === VALIDATION PHASE ===
    model.eval()  # Disable dropout, batchnorm
    val_loss = 0
    with torch.no_grad():  # No gradients = faster
        for Xb, yb in val_loader:
            preds = model(Xb)
            loss = criterion(preds, yb)
            val_loss += loss.item()

    # Compute average
    val_mse = val_loss / len(val_loader)

    # Early stopping check
    if early_stop(val_mse):
      print("Early stopping!")
      break

    # Average losses
    train_losses.append(train_loss / len(train_loader))
    val_losses.append(val_mse)


    # Print every 5 epochs
    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | Train MSE: {train_losses[-1]:.4f} | Val MSE: {val_losses[-1]:.4f}")

print("Training complete!")


Epoch   0 | Train MSE: 6.9663 | Val MSE: 1.3808
Epoch   5 | Train MSE: 1.5447 | Val MSE: 1.1098
Epoch  10 | Train MSE: 1.3857 | Val MSE: 1.1782
Epoch  15 | Train MSE: 1.3022 | Val MSE: 1.1295
Epoch  20 | Train MSE: 1.2749 | Val MSE: 1.1621
Epoch  25 | Train MSE: 1.2590 | Val MSE: 1.1337
Epoch  30 | Train MSE: 1.2383 | Val MSE: 1.0834
Early stopping!
Training complete!


In [99]:
import torch
from sklearn.metrics import r2_score

# Set model to evaluation mode
model.eval()

# Make predictions (no gradients needed)
with torch.no_grad():
    X_val_tensor = torch.FloatTensor(X_val)  # Convert to tensor if needed
    predictions = model(X_val_tensor).cpu().numpy().flatten()
    #predictions = model(X_val_tensor).numpy()  # Get predictions as numpy

y_val_flat = y_val.flatten()
# Calculate accuracy
r2 = r2_score(y_val_flat, predictions)

print(f"🎯 Final R² Score: {r2:.3f} ({r2*100:.1f}% explained variance)")
#print(f"🎯 Final R² Score: 0.725 (72.5% accurate)")  # Your result!
print("\nFirst 10 predictions:")
for i in range(min(10, len(y_val_flat))):
    real = float(y_val[i])
    pred = float(predictions[i])
    print(f"Real: {real:.1f} → Predicted: {pred:.1f} (Error: {abs(real - pred):.2f})")

🎯 Final R² Score: 0.733 (73.3% explained variance)

First 10 predictions:
Real: 3.0 → Predicted: 3.7 (Error: 0.68)
Real: 7.0 → Predicted: 7.4 (Error: 0.43)
Real: 5.0 → Predicted: 4.1 (Error: 0.86)
Real: 3.0 → Predicted: 3.1 (Error: 0.10)
Real: 2.0 → Predicted: 3.9 (Error: 1.93)
Real: 6.0 → Predicted: 6.4 (Error: 0.40)
Real: 4.0 → Predicted: 3.4 (Error: 0.63)
Real: 7.0 → Predicted: 6.6 (Error: 0.41)
Real: 4.0 → Predicted: 4.1 (Error: 0.13)
Real: 7.0 → Predicted: 6.5 (Error: 0.54)


In [100]:
import torch
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Step 8: Final evaluation on the TEST set ──────────────────────────────────

model.eval()  # Put the model in evaluation mode (no dropout, no batchnorm updates)

test_dataset = TabularDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256)

# Lists to store all predictions and true values
all_preds, all_targets = [], []
test_loss = 0.0  # To track average MSE loss (using PyTorch loss function)

with torch.no_grad():  # No gradients needed during evaluation
    for Xb, yb in test_loader:  # Go through the test data in batches
        preds = model(Xb)              # Make predictions for the batch
        loss = criterion(preds, yb)    # Compute MSE loss for this batch
        test_loss += loss.item()       # Add to total loss

        # Save results for later analysis
        all_preds.append(preds.cpu().numpy())
        all_targets.append(yb.cpu().numpy())

# Combine batches into full arrays
preds_np = np.concatenate(all_preds).flatten()
targets_np = np.concatenate(all_targets).flatten()

# ── Compute overall metrics ───────────────────────────────────────────────────
mse = mean_squared_error(targets_np, preds_np)  # Average squared difference
rmse = np.sqrt(mse)                             # Root mean squared error
mae = mean_absolute_error(targets_np, preds_np) # Average absolute difference
r2 = r2_score(targets_np, preds_np)             # R²: how well model explains variance

# Also compute the mean PyTorch loss across all batches (for comparison)
avg_test_loss = test_loss / len(test_loader)

print(f"Average test MSE (from PyTorch): {avg_test_loss:.4f}")
print(f"MSE  (from sklearn): {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

# ── Show some sample predictions ──────────────────────────────────────────────
print("\nFirst 10 predictions (real vs. predicted stress level):")
for i in range(min(10, len(targets_np))):
    error = abs(targets_np[i] - preds_np[i])
    print(f"Real: {targets_np[i]:.2f} | Predicted: {preds_np[i]:.2f} | Error: {error:.2f}")

# ── Check residual distribution ───────────────────────────────────────────────
residuals = targets_np - preds_np  # Difference between real and predicted
print("\nResidual mean:", residuals.mean())  # Should be near zero
print("Residual std :", residuals.std())     # Spread of prediction errors


Average test MSE (from PyTorch): 1.1081
MSE  (from sklearn): 1.1065
RMSE : 1.0519
MAE  : 0.8461
R²   : 0.7337

First 10 predictions (real vs. predicted stress level):
Real: 9.00 | Predicted: 8.67 | Error: 0.33
Real: 6.00 | Predicted: 5.62 | Error: 0.38
Real: 6.00 | Predicted: 6.14 | Error: 0.14
Real: 3.00 | Predicted: 3.54 | Error: 0.54
Real: 4.00 | Predicted: 4.49 | Error: 0.49
Real: 10.00 | Predicted: 8.25 | Error: 1.75
Real: 10.00 | Predicted: 9.36 | Error: 0.64
Real: 7.00 | Predicted: 7.12 | Error: 0.12
Real: 2.00 | Predicted: 3.60 | Error: 1.60
Real: 8.00 | Predicted: 7.02 | Error: 0.98

Residual mean: 0.07973104
Residual std : 1.0488864
